In [ ]:
'''
Spark DataFrame and SQL Analysis​
Load a dataset from Kaggle or another relevant source into a Spark DataFrame.
Then perform SQL-based analysis, including filtering, grouping, joining, and aggregations with pyspark.sql.functions. ​
​'''

'\nSpark DataFrame and SQL Analysis\u200b\nLoad a dataset from Kaggle or another relevant source into a Spark DataFrame.\nThen perform SQL-based analysis, including filtering, grouping, joining, and aggregations with pyspark.sql.functions. \u200b\n\u200b'

In [ ]:
import yfinance as yf
import pandas as pd
import datetime

#from pyspark import SparkContext
#sc = SparkContext.getOrCreate()
from pyspark import SparkContext
from pyspark.sql import SparkSession
import pyspark
from pyspark.sql.window import Window
from pyspark.sql.functions import substring, col, year, lag

In [ ]:
spark = SparkSession.builder.getOrCreate()

### Load data about inflation

In [ ]:
inflation_df = spark.read.csv('/content/FPCPITOTLZGUSA.csv', header=True, inferSchema=True)
inflation_df.collect()

[Row(observation_date=datetime.date(1990, 1, 1), FPCPITOTLZGUSA=5.39795643990325),
 Row(observation_date=datetime.date(1991, 1, 1), FPCPITOTLZGUSA=4.23496396453849),
 Row(observation_date=datetime.date(1992, 1, 1), FPCPITOTLZGUSA=3.02881967814969),
 Row(observation_date=datetime.date(1993, 1, 1), FPCPITOTLZGUSA=2.95165696638559),
 Row(observation_date=datetime.date(1994, 1, 1), FPCPITOTLZGUSA=2.60744159215453),
 Row(observation_date=datetime.date(1995, 1, 1), FPCPITOTLZGUSA=2.80541968853662),
 Row(observation_date=datetime.date(1996, 1, 1), FPCPITOTLZGUSA=2.93120419993441),
 Row(observation_date=datetime.date(1997, 1, 1), FPCPITOTLZGUSA=2.33768993730735),
 Row(observation_date=datetime.date(1998, 1, 1), FPCPITOTLZGUSA=1.55227909874364),
 Row(observation_date=datetime.date(1999, 1, 1), FPCPITOTLZGUSA=2.18802719697358),
 Row(observation_date=datetime.date(2000, 1, 1), FPCPITOTLZGUSA=3.37685727149929),
 Row(observation_date=datetime.date(2001, 1, 1), FPCPITOTLZGUSA=2.82617111885407),
 Row

In [ ]:
inflation_df = inflation_df.withColumnRenamed('FPCPITOTLZGUSA', 'Inflation')

In [ ]:
# Stock prices of selected companies

tickers = ['IBM', 'JNJ', 'JPM', 'INTC']
companies = ['IBM', 'Johnson & Johnson', 'JPMorgan', 'Intel']

end_date = datetime.datetime(2025, 1, 1)
start_date = datetime.datetime(1990, 1, 1)

# Downlaod data for the given period of time
stock_prices = yf.download(tickers, start=start_date, auto_adjust=False, end=end_date, interval='1mo')

stock_prices_df = spark.createDataFrame(stock_prices.reset_index())

[*********************100%***********************]  4 of 4 completed


### Computing annualsied average monthly returns and average monthly stock prices

In [ ]:
stock_prices_df.show()

+-------------------+--------------------+---------------------+--------------------+--------------------+------------------+------------------+----------------+------------------+------------------+------------------+---------------+------------------+------------------+------------------+--------------+------------------+------------------+------------------+---------------+------------------+-----------------+------------------+-----------------+-----------------+
|       ('Date', '')|('Adj Close', 'IBM')|('Adj Close', 'INTC')|('Adj Close', 'JNJ')|('Adj Close', 'JPM')|  ('Close', 'IBM')| ('Close', 'INTC')|('Close', 'JNJ')|  ('Close', 'JPM')|   ('High', 'IBM')|  ('High', 'INTC')|('High', 'JNJ')|   ('High', 'JPM')|    ('Low', 'IBM')|   ('Low', 'INTC')|('Low', 'JNJ')|    ('Low', 'JPM')|   ('Open', 'IBM')|  ('Open', 'INTC')|('Open', 'JNJ')|   ('Open', 'JPM')|('Volume', 'IBM')|('Volume', 'INTC')|('Volume', 'JNJ')|('Volume', 'JPM')|
+-------------------+--------------------+--------------

In [ ]:
# Add column about Year

stock_prices_df_with_year = stock_prices_df.withColumn("Year", year(col("('Date', '')")))

In [ ]:
stock_prices_df_with_year.show()

+-------------------+--------------------+---------------------+--------------------+--------------------+------------------+------------------+----------------+------------------+------------------+------------------+---------------+------------------+------------------+------------------+--------------+------------------+------------------+------------------+---------------+------------------+-----------------+------------------+-----------------+-----------------+----+
|       ('Date', '')|('Adj Close', 'IBM')|('Adj Close', 'INTC')|('Adj Close', 'JNJ')|('Adj Close', 'JPM')|  ('Close', 'IBM')| ('Close', 'INTC')|('Close', 'JNJ')|  ('Close', 'JPM')|   ('High', 'IBM')|  ('High', 'INTC')|('High', 'JNJ')|   ('High', 'JPM')|    ('Low', 'IBM')|   ('Low', 'INTC')|('Low', 'JNJ')|    ('Low', 'JPM')|   ('Open', 'IBM')|  ('Open', 'INTC')|('Open', 'JNJ')|   ('Open', 'JPM')|('Volume', 'IBM')|('Volume', 'INTC')|('Volume', 'JNJ')|('Volume', 'JPM')|Year|
+-------------------+--------------------+----

In [ ]:
# Add column with returns (a change in price)
window = Window.orderBy("('Date', '')")

for ticker in tickers:
    col_name = f"('Adj Close', '{ticker}')"
    return_col = f"Return_{ticker}"

    # Values of column col_name with a shift 1 (consists of previous values)
    # For instance:
    # Date    Adj_Close    Previous (this is lag)
    # 10.02      15               None
    # 11.02      16                15
    prev = lag(col(col_name)).over(window)

    stock_prices_df_with_year = stock_prices_df_with_year.withColumn(return_col, (col(col_name) - prev) / prev)

In [ ]:
stock_prices_df_with_year.show()

+-------------------+--------------------+---------------------+--------------------+--------------------+------------------+------------------+----------------+------------------+------------------+------------------+---------------+------------------+------------------+------------------+--------------+------------------+------------------+------------------+---------------+------------------+-----------------+------------------+-----------------+-----------------+----+--------------------+--------------------+--------------------+--------------------+
|       ('Date', '')|('Adj Close', 'IBM')|('Adj Close', 'INTC')|('Adj Close', 'JNJ')|('Adj Close', 'JPM')|  ('Close', 'IBM')| ('Close', 'INTC')|('Close', 'JNJ')|  ('Close', 'JPM')|   ('High', 'IBM')|  ('High', 'INTC')|('High', 'JNJ')|   ('High', 'JPM')|    ('Low', 'IBM')|   ('Low', 'INTC')|('Low', 'JNJ')|    ('Low', 'JPM')|   ('Open', 'IBM')|  ('Open', 'INTC')|('Open', 'JNJ')|   ('Open', 'JPM')|('Volume', 'IBM')|('Volume', 'INTC')|('Vo

In [ ]:
col_names = ["('Date', '')", "Year"]
adj_close_names = [f"('Adj Close', '{t}')" for t in tickers]
return_names = [f"Return_{t}" for t in tickers]
col_names.extend(adj_close_names)
col_names.extend(return_names)

# just creating a copy of stock_prices_df_with_year
stock_prices_df_final = stock_prices_df_with_year.select(col_names)

In [ ]:
stock_prices_df_final.show()

+-------------------+----+--------------------+--------------------+--------------------+---------------------+--------------------+--------------------+--------------------+--------------------+
|       ('Date', '')|Year|('Adj Close', 'IBM')|('Adj Close', 'JNJ')|('Adj Close', 'JPM')|('Adj Close', 'INTC')|          Return_IBM|          Return_JNJ|          Return_JPM|         Return_INTC|
+-------------------+----+--------------------+--------------------+--------------------+---------------------+--------------------+--------------------+--------------------+--------------------+
|1990-01-01 00:00:00|1990|   9.471813201904297|  2.7706708908081055|  2.7397866249084473|   0.6882469058036804|                NULL|                NULL|                NULL|                NULL|
|1990-02-01 00:00:00|1990|    9.97601318359375|  2.8367996215820312|   2.726486921310425|   0.7013151049613953| 0.05323162217642598|0.023867407346470658|-0.00485428444577...|0.018987661328393236|
|1990-03-01 00:00:00

In [ ]:
stock_prices_df_final = stock_prices_df_final.withColumnRenamed("('Date', '')", "Date")
for i, adj_close_col in enumerate(adj_close_names):
    stock_prices_df_final = stock_prices_df_final.withColumnRenamed(adj_close_col, f'Adj_Close_{tickers[i]}')

In [ ]:
for ticker in tickers:
    return_col = f"Return_{ticker}"
    # Annualize returns by multiplying them by 12
    stock_prices_df_final = stock_prices_df_final.withColumn(return_col, 12 * col(return_col))

In [ ]:
aggregations = dict()
for t in tickers:
    aggregations[f'Adj_Close_{t}'] = "avg"
    aggregations[f"Return_{t}"] = "avg"

In [ ]:
# Aggregate all rows and count their average (monthly average stock price, monthly averaged annualised returns)
stock_prices_df_final = stock_prices_df_final.groupBy("Year").agg(aggregations).sort("Year")

In [ ]:
stock_prices_df_final.show()

+----+--------------------+------------------+--------------------+------------------+-------------------+------------------+--------------------+--------------------+
|Year|     avg(Return_JNJ)|avg(Adj_Close_IBM)|    avg(Return_INTC)|avg(Adj_Close_JNJ)|avg(Adj_Close_INTC)|avg(Adj_Close_JPM)|     avg(Return_IBM)|     avg(Return_JPM)|
+----+--------------------+------------------+--------------------+------------------+-------------------+------------------+--------------------+--------------------+
|1990| 0.38737605998234215|10.691346963246664|0.048682584990028796|3.3660861452420554| 0.6924214363098145| 2.225646158059438|  0.2169555092429036| -0.7001865096066288|
|1991|  0.5312115299335166|10.732670545578003| 0.31399239560391184| 5.002179066340129|   0.81475197772185|2.5442809561888375|-0.16305071209094169|  0.8411002716767025|
|1992|-0.09244804643748526| 8.861131946245829|  0.6522827070786116| 5.449494520823161| 1.0873449941476185| 4.308632890383403| -0.4427957300482494|  0.7181187458

In [ ]:
for t in tickers:
    return_name = f"avg(Return_{t})"
    pct_return_name = f"Return_avg_pct_{t}"
    stock_prices_df_final = stock_prices_df_final.withColumn(pct_return_name, 100 * col(return_name))

### Computing quantiles

In [ ]:
qs = [0.25, 0.5, 0.75]

In [ ]:
all_quantiles = stock_prices_df_final.approxQuantile([f"Return_avg_pct_{t}" for t in tickers], qs, 0.01)

In [ ]:
all_quantiles

[[-11.124501309302026, 18.36935137721978, 25.398440074940215],
 [-1.0636387124666626, 12.669691971684824, 22.60805125250277],
 [-4.954622783728746, 17.274696674411945, 38.49306611449599],
 [-7.682136075277497, 16.909292903225047, 40.68842487300384]]

In [ ]:
# creating a copy of stock_prices_df_final DataFrame
stock_prices_df_final_q75 = stock_prices_df_final.select('*')
q75_idx = 2
for idx, t in enumerate(tickers):
    # Iteratively looking for an upper quartile of all companies' returns
    pct_return_name = f"Return_avg_pct_{t}"
    stock_prices_df_final_q75 = stock_prices_df_final_q75.filter(col(pct_return_name) > all_quantiles[idx][q75_idx])

stock_prices_df_final_q75.show()

+----+-------------------+------------------+------------------+------------------+-------------------+------------------+-------------------+------------------+------------------+------------------+------------------+-------------------+
|Year|    avg(Return_JNJ)|avg(Adj_Close_IBM)|  avg(Return_INTC)|avg(Adj_Close_JNJ)|avg(Adj_Close_INTC)|avg(Adj_Close_JPM)|    avg(Return_IBM)|   avg(Return_JPM)|Return_avg_pct_IBM|Return_avg_pct_JNJ|Return_avg_pct_JPM|Return_avg_pct_INTC|
+----+-------------------+------------------+------------------+------------------+-------------------+------------------+-------------------+------------------+------------------+------------------+------------------+-------------------+
|1995|0.48474542666555515| 10.83860675493876| 0.649462153737033| 8.319660822550455|  3.883302370707194| 6.989015261332194|0.26027775357521654|0.5731128632310751|26.027775357521655|48.474542666555514| 57.31128632310751|   64.9462153737033|
|1998| 0.2831009543265617| 30.83342742919922

In [ ]:
stock_prices_df_final_q75.show()

+----+-------------------+------------------+------------------+------------------+-------------------+------------------+-------------------+------------------+------------------+------------------+------------------+-------------------+
|Year|    avg(Return_JNJ)|avg(Adj_Close_IBM)|  avg(Return_INTC)|avg(Adj_Close_JNJ)|avg(Adj_Close_INTC)|avg(Adj_Close_JPM)|    avg(Return_IBM)|   avg(Return_JPM)|Return_avg_pct_IBM|Return_avg_pct_JNJ|Return_avg_pct_JPM|Return_avg_pct_INTC|
+----+-------------------+------------------+------------------+------------------+-------------------+------------------+-------------------+------------------+------------------+------------------+------------------+-------------------+
|1995|0.48474542666555515| 10.83860675493876| 0.649462153737033| 8.319660822550455|  3.883302370707194| 6.989015261332194|0.26027775357521654|0.5731128632310751|26.027775357521655|48.474542666555514| 57.31128632310751|   64.9462153737033|
|1998| 0.2831009543265617| 30.83342742919922

In [ ]:
# creating a copy of stock_prices_df_final DataFrame

stock_prices_df_final_q25 = stock_prices_df_final.select('*')
q25_idx = 0
for idx, t in enumerate(tickers):
    # Iteratively looking for a lower quartile of all companies' returns
    pct_return_name = f"Return_avg_pct_{t}"
    stock_prices_df_final_q25 = stock_prices_df_final_q25.filter(col(pct_return_name) <= all_quantiles[idx][q25_idx])

stock_prices_df_final_q25.show()

+----+--------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+--------------------+------------------+------------------+-------------------+-------------------+
|Year|     avg(Return_JNJ)|avg(Adj_Close_IBM)|   avg(Return_INTC)|avg(Adj_Close_JNJ)|avg(Adj_Close_INTC)|avg(Adj_Close_JPM)|    avg(Return_IBM)|     avg(Return_JPM)|Return_avg_pct_IBM|Return_avg_pct_JNJ| Return_avg_pct_JPM|Return_avg_pct_INTC|
+----+--------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+--------------------+------------------+------------------+-------------------+-------------------+
|2002|-0.05899633487331318| 40.96042251586914|-0.4883436692179373|30.076281706492107| 12.921486258506775|14.548853635787964|-0.3027620229749906|-0.21012912894333424|-30.27620229749906|-5.899633487331318|-21.012912894333425| -48.83436692179373|
|2008|-0.065211133693954

In [ ]:
inflation_df.show()

+----------------+------------------+
|observation_date|         Inflation|
+----------------+------------------+
|      1990-01-01|  5.39795643990325|
|      1991-01-01|  4.23496396453849|
|      1992-01-01|  3.02881967814969|
|      1993-01-01|  2.95165696638559|
|      1994-01-01|  2.60744159215453|
|      1995-01-01|  2.80541968853662|
|      1996-01-01|  2.93120419993441|
|      1997-01-01|  2.33768993730735|
|      1998-01-01|  1.55227909874364|
|      1999-01-01|  2.18802719697358|
|      2000-01-01|  3.37685727149929|
|      2001-01-01|  2.82617111885407|
|      2002-01-01|  1.58603162650601|
|      2003-01-01|  2.27009497336115|
|      2004-01-01|  2.67723669309172|
|      2005-01-01|   3.3927468454955|
|      2006-01-01|  3.22594410070404|
|      2007-01-01|  2.85267248150138|
|      2008-01-01|    3.839100296651|
|      2009-01-01|-0.355546266299747|
+----------------+------------------+
only showing top 20 rows


In [ ]:
# Computing quantiles of inflation
inflation_q = inflation_df.approxQuantile('Inflation', [0.25, 0.75], 0.01)

In [ ]:
inflation_q

[1.6400434423899, 3.22594410070404]

In [ ]:
# Looking for such rows where inflation exceeded an upper quartile
inflation_q75_df = inflation_df.filter(col('Inflation') >= inflation_q[1])

In [ ]:
inflation_q75_df.show()

+----------------+----------------+
|observation_date|       Inflation|
+----------------+----------------+
|      1990-01-01|5.39795643990325|
|      1991-01-01|4.23496396453849|
|      2000-01-01|3.37685727149929|
|      2005-01-01| 3.3927468454955|
|      2006-01-01|3.22594410070404|
|      2008-01-01|  3.839100296651|
|      2021-01-01|4.69785886363742|
|      2022-01-01|8.00279982052121|
|      2023-01-01|4.11633838374488|
+----------------+----------------+



In [ ]:
inflation_q75_df = inflation_q75_df.withColumn('Year', year(col('observation_date')))

In [ ]:
# Looking for such rows where inflation was lower than a lower quartile
inflation_q25_df = inflation_df.filter(col('Inflation') <= inflation_q[0])

In [ ]:
inflation_q25_df.show()

+----------------+------------------+
|observation_date|         Inflation|
+----------------+------------------+
|      1998-01-01|  1.55227909874364|
|      2002-01-01|  1.58603162650601|
|      2009-01-01|-0.355546266299747|
|      2010-01-01|   1.6400434423899|
|      2013-01-01|  1.46483265562717|
|      2014-01-01|  1.62222297740817|
|      2015-01-01| 0.118627135552451|
|      2016-01-01|  1.26158320570536|
|      2020-01-01|  1.23358439630629|
+----------------+------------------+



### Executing JOINS

In [ ]:
# Performing an inner join of stock prices of lower quartile AND inflation rates of an upper quartile
# looking for years where inflation was high and stock prices low
stock_prices_df_final_q25.join(inflation_q75_df, stock_prices_df_final_q25.Year == inflation_q75_df.Year, "inner").show()

+----+--------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+-------------------+------------------+------------------+------------------+-------------------+----------------+--------------+----+
|Year|     avg(Return_JNJ)|avg(Adj_Close_IBM)|   avg(Return_INTC)|avg(Adj_Close_JNJ)|avg(Adj_Close_INTC)|avg(Adj_Close_JPM)|    avg(Return_IBM)|    avg(Return_JPM)|Return_avg_pct_IBM|Return_avg_pct_JNJ|Return_avg_pct_JPM|Return_avg_pct_INTC|observation_date|     Inflation|Year|
+----+--------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+-------------------+------------------+------------------+------------------+-------------------+----------------+--------------+----+
|2008|-0.06521113369395494| 58.41475168863932|-0.4944807632091249| 38.27710851033529| 12.187598625818888|25.819751580556233|-0.1857094140044433|-0.1672039183558767

In [ ]:
inflation_q25_df = inflation_q25_df.withColumn('Year', year(col('observation_date')))

In [ ]:
# Performing an inner join of stock prices of upper quartile AND inflation rates of an lower quartile
# looking for years where inflation was low and stock prices high
stock_prices_df_final_q75.join(inflation_q25_df, stock_prices_df_final_q75.Year == inflation_q25_df.Year, "inner").show()

+----+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+----------------+----------------+----+
|Year|   avg(Return_JNJ)|avg(Adj_Close_IBM)|  avg(Return_INTC)|avg(Adj_Close_JNJ)|avg(Adj_Close_INTC)|avg(Adj_Close_JPM)|   avg(Return_IBM)|   avg(Return_JPM)|Return_avg_pct_IBM|Return_avg_pct_JNJ|Return_avg_pct_JPM|Return_avg_pct_INTC|observation_date|       Inflation|Year|
+----+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+----------------+----------------+----+
|1998|0.2831009543265617| 30.83342742919922|0.6243915035935702|18.628411293029785| 12.181382894515991|19.126561244328816|0.6393482854419047|0.4382823363406825| 63.934828544

In [ ]:
# Performing an inner join of stock prices of upper quartile AND inflation rates of an upper quartile
# looking for years where inflation was high and stock prices high
stock_prices_df_final_q75.join(inflation_q75_df, stock_prices_df_final_q75.Year == inflation_q75_df.Year, "inner").show()

+----+---------------+------------------+----------------+------------------+-------------------+------------------+---------------+---------------+------------------+------------------+------------------+-------------------+----------------+---------+----+
|Year|avg(Return_JNJ)|avg(Adj_Close_IBM)|avg(Return_INTC)|avg(Adj_Close_JNJ)|avg(Adj_Close_INTC)|avg(Adj_Close_JPM)|avg(Return_IBM)|avg(Return_JPM)|Return_avg_pct_IBM|Return_avg_pct_JNJ|Return_avg_pct_JPM|Return_avg_pct_INTC|observation_date|Inflation|Year|
+----+---------------+------------------+----------------+------------------+-------------------+------------------+---------------+---------------+------------------+------------------+------------------+-------------------+----------------+---------+----+
+----+---------------+------------------+----------------+------------------+-------------------+------------------+---------------+---------------+------------------+------------------+------------------+-------------------+-

In [ ]:
# Performing an inner join of stock prices of lower quartile AND inflation rates of an lower quartile
# looking for years where inflation was low and stock prices low
stock_prices_df_final_q25.join(inflation_q25_df, stock_prices_df_final_q25.Year == inflation_q25_df.Year, "inner").show()

+----+--------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+--------------------+------------------+------------------+-------------------+-------------------+----------------+----------------+----+
|Year|     avg(Return_JNJ)|avg(Adj_Close_IBM)|   avg(Return_INTC)|avg(Adj_Close_JNJ)|avg(Adj_Close_INTC)|avg(Adj_Close_JPM)|    avg(Return_IBM)|     avg(Return_JPM)|Return_avg_pct_IBM|Return_avg_pct_JNJ| Return_avg_pct_JPM|Return_avg_pct_INTC|observation_date|       Inflation|Year|
+----+--------------------+------------------+-------------------+------------------+-------------------+------------------+-------------------+--------------------+------------------+------------------+-------------------+-------------------+----------------+----------------+----+
|2002|-0.05899633487331318| 40.96042251586914|-0.4883436692179373|30.076281706492107| 12.921486258506775|14.548853635787964|-0.3027620229749906|-0.2101